# MedRAX Premium — Benchmark Evaluation

**Pipeline**: ReAct Agent → Multi-Tool Execution → Canonical Output → Conflict Detection (BERT NLI + Rule-Based + GACL) → Conflict Resolution (Argumentation Graph + Tool Trust + Abstention Logic)

**Benchmark**: ChestAgentBench (MCQ on chest X-rays)

**Environment**: Kaggle 2×T4 GPUs, 4-bit quantization

In [1]:
# ===============================
# 📦 Cell 1 — Install Dependencies
# ===============================

# Install lightweight packages (no torch upgrade)
%pip install -q \
    langchain-openai \
    langchain-core \
    langchain-community \
    langgraph \
    tenacity \
    python-dotenv \
    accelerate \
    sentencepiece \
    timm \
    einops \
    shortuuid \
    gradio \
    torchxrayvision \
    scikit-image \
    pydicom

# Install transformers WITHOUT touching torch/torchvision
%pip install -q --no-deps "transformers>=4.40.0" tokenizers huggingface-hub safetensors

# bitsandbytes needs its own install (GPU-specific build)
%pip install -q bitsandbytes

print("✅ All packages installed — no restart needed")

# Robust Kaggle path fix: ensure medrax_premium is importable before Cell 3
import sys
from pathlib import Path

SEARCH_BASES = [Path("/kaggle/working"), Path("/kaggle/input"), Path.cwd()]
MAX_DEPTH = 6


def _depth_ok(base: Path, p: Path, max_depth: int) -> bool:
    try:
        rel = p.relative_to(base)
        return len(rel.parts) <= max_depth
    except Exception:
        return False


def _find_project_root() -> Path | None:
    # Fast-path common locations first
    common = [
        Path("/kaggle/working/Medrax_premium"),
        Path("/kaggle/input/medrax-premium/Medrax_premium"),
        Path("/kaggle/input/datasets/mohammadninadmahmud/medrax-premium/Medrax_premium"),
    ]
    for c in common:
        if (c / "medrax_premium" / "__init__.py").is_file():
            return c

    # Generic search: locate package marker and return its parent as repo root
    for base in SEARCH_BASES:
        if not base.exists():
            continue
        for init_file in base.rglob("medrax_premium/__init__.py"):
            root = init_file.parent.parent
            if _depth_ok(base, root, MAX_DEPTH):
                return root
    return None


ROOT_DISCOVERED = _find_project_root()
if ROOT_DISCOVERED is None:
    raise FileNotFoundError(
        "Could not find project root containing medrax_premium/__init__.py under /kaggle/input or /kaggle/working."
    )

if str(ROOT_DISCOVERED) not in sys.path:
    sys.path.insert(0, str(ROOT_DISCOVERED))

print(f"✅ medrax_premium import path ready: {ROOT_DISCOVERED}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 13.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 57.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyte

In [2]:
# ===============================
# 📦 Cell 2 — Core Imports
# ===============================

import operator
import warnings
from typing import *
import traceback
import os
import torch
import json
import glob
import time
import re
import gc
import sys
import uuid
import logging
from datetime import datetime
from pathlib import Path
from collections import defaultdict, Counter

from dotenv import load_dotenv
from IPython.display import Image
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from tenacity import retry, wait_exponential, stop_after_attempt
import matplotlib.pyplot as plt
import numpy as np

# ── Transformers compatibility shim (Kaggle runtime) ──
import transformers.pytorch_utils as _ptu
if not hasattr(_ptu, "find_pruneable_heads_and_indices"):
    def _find_pruneable(heads, n_heads, head_size, already_pruned):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned
        for h in heads:
            h -= sum(1 if ph < h else 0 for ph in already_pruned)
            mask[h] = 0
        return heads, torch.arange(mask.numel())[mask.view(-1).contiguous().eq(1)].long()
    _ptu.find_pruneable_heads_and_indices = _find_pruneable
if not hasattr(_ptu, "prune_linear_layer"):
    import torch.nn as _nn
    def _prune_linear(layer, index, dim=0):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        b = None
        if layer.bias is not None:
            b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
        new = _nn.Linear(W.size(1), W.size(0), bias=b is not None).to(layer.weight.device)
        new.weight.requires_grad_(False).copy_(W).requires_grad_(True)
        if b is not None:
            new.bias.requires_grad_(False).copy_(b).requires_grad_(True)
        return new
    _ptu.prune_linear_layer = _prune_linear

# ===============================
# 🔑 API Keys — Kaggle Secrets → os.environ
# ===============================
# On Kaggle: add OPENAI_API_KEY + OPENAI_BASE_URL in the Secrets panel
# (right sidebar) and enable "Notebook access" for both.
# GitHub PATs require base_url="https://models.inference.ai.azure.com"
# load_dotenv() handles local .env files for non-Kaggle runs.

load_dotenv()  # picks up .env locally (no-op on Kaggle)

try:
    from kaggle_secrets import UserSecretsClient as _KSC
    _ks = _KSC()
    for _sname in ["OPENAI_API_KEY", "OPENAI_BASE_URL"]:
        try:
            _sval = _ks.get_secret(_sname)
            if _sval:
                os.environ[_sname] = _sval
                _display = _sval[:8] + "..." if len(_sval) > 8 else "***"
                print(f"  ✅ Kaggle secret loaded: {_sname} = {_display}")
        except Exception:
            pass  # secret not defined in panel — skip silently
except ImportError:
    pass  # not running on Kaggle

# Hard fail early if no key at all — saves confusion later
_api_key = os.environ.get("OPENAI_API_KEY", "")
if not _api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set!\n"
        "  • On Kaggle: add it in the Secrets panel (right sidebar) and enable 'Notebook access'\n"
        "  • Locally: add OPENAI_API_KEY=sk-... to your .env file"
    )

# ── GitHub Models auto-detection ──────────────────────────────────────
# A GitHub PAT (starts with "github_" or "ghp_") only works against the
# GitHub Models inference endpoint — NOT api.openai.com.
# Auto-set OPENAI_BASE_URL if the user forgot to add it as a secret.
_GITHUB_MODELS_URL = "https://models.inference.ai.azure.com"
_is_github_pat = _api_key.startswith(("github_", "ghp_"))
if _is_github_pat and not os.environ.get("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = _GITHUB_MODELS_URL
    print(f"  🔀 GitHub PAT detected → OPENAI_BASE_URL auto-set to {_GITHUB_MODELS_URL}")

_base_url = os.environ.get("OPENAI_BASE_URL", "")
print(f"  ✅ OPENAI_API_KEY ready ({_api_key[:10]}...)")
if _base_url:
    print(f"  ✅ OPENAI_BASE_URL = {_base_url}")
else:
    print(f"  ℹ️  No OPENAI_BASE_URL — using OpenAI directly (api.openai.com)")

# ===============================
# 🔧 Kaggle ROOT + Path Detection
# ===============================

INPUT = Path("/kaggle/input")
WORK  = Path("/kaggle/working")


def _discover_base_dir() -> Path:
    preferred = Path("/kaggle/input/datasets/mohammadninadmahmud")
    if preferred.is_dir():
        return preferred

    hubs = [INPUT / "datasets", INPUT]
    for hub in hubs:
        if not hub.is_dir():
            continue
        for candidate in hub.glob("*"):
            if not candidate.is_dir():
                continue
            probe = candidate / "medrax-premium" / "Medrax_premium" / "medrax_premium" / "__init__.py"
            if probe.is_file():
                return candidate

    return INPUT


_B = _discover_base_dir()
ROOT           = _B / "medrax-premium"       / "Medrax_premium"
BENCH_ROOT     = _B / "chestagentbench"      / "chestagentbench"
MODELS_CHEX    = _B / "models-chexagent"     / "models-chexagent"
MODELS_CORE    = _B / "models-core"          / "models-core"
MODELS_LLAVA   = _B / "models-llava-med"     / "models-llava-med"
MODELS_MEDTOOL = _B / "models-medical-tools" / "models-medical-tools"

# Fallback if medrax-premium directory naming is different.
if not (ROOT / "medrax_premium" / "__init__.py").is_file():
    fallback_roots = [
        WORK / "Medrax_premium",
        INPUT / "medrax-premium" / "Medrax_premium",
    ]
    for hub in [INPUT / "datasets", INPUT]:
        if not hub.is_dir():
            continue
        fallback_roots.extend(hub.glob("*/Medrax_premium"))
        fallback_roots.extend(hub.glob("*/*/Medrax_premium"))

    for candidate in fallback_roots:
        if (candidate / "medrax_premium" / "__init__.py").is_file():
            ROOT = candidate
            break

# ── Add ROOT to path BEFORE importing medrax_premium ──
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ── Import medrax_premium ──
from medrax_premium.agent import *
from medrax_premium.tools import *
from medrax_premium.utils import *

warnings.filterwarnings("ignore")

# ===============================
# 📂 Directory Configuration
# ===============================

PROMPT_FILE    = ROOT / "medrax_premium" / "docs" / "system_prompts.txt"
METADATA_FILE  = BENCH_ROOT / "metadata.jsonl"
FIGURES_DIR    = BENCH_ROOT / "figures"

# Tell agent tools where benchmark images live (for relative path resolution)
os.environ["MEDRAX_FIGURES_DIR"] = str(BENCH_ROOT)

paths = {
    "ROOT": ROOT, "BENCH_ROOT": BENCH_ROOT,
    "MODELS_CHEX": MODELS_CHEX, "MODELS_CORE": MODELS_CORE,
    "MODELS_LLAVA": MODELS_LLAVA, "MODELS_MEDTOOL": MODELS_MEDTOOL,
}
for k, v in paths.items():
    print(f"  {'✅' if v.is_dir() else '❌'} {k} → {v}")

# ===============================
# 🗂️ Logging Setup
# ===============================

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

model_name  = "medrax_premium"
temperature = 0.2
DEVICE_0    = "cuda:0"
DEVICE_1    = "cuda:1"

print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
for i in range(torch.cuda.device_count()):
    total = torch.cuda.mem_get_info(i)[1] / 1024**3
    print(f"  GPU-{i}: {torch.cuda.get_device_name(i)} — {total:.1f} GB")

LOG_DIR  = WORK / "medrax_premium_logs"
TEMP_DIR = WORK / "temp"
ACL_DIR  = WORK / "acl_submission"
for d in (LOG_DIR, TEMP_DIR, ACL_DIR):
    d.mkdir(parents=True, exist_ok=True)

log_filename = LOG_DIR / f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

logging.basicConfig(
    filename=str(log_filename),
    level=logging.INFO,
    format="%(message)s",
    force=True,
)


print(f"Logging to: {log_filename}")
print(f"Target layout — GPU-0: report+cls+seg+LLaVA-Med ≈5.5GB | GPU-1: CheXagent ≈5GB (4-bit)")

  ✅ Kaggle secret loaded: OPENAI_API_KEY = github_p...
  ✅ Kaggle secret loaded: OPENAI_BASE_URL = https://...
  ✅ OPENAI_API_KEY ready (github_pat...)
  ✅ OPENAI_BASE_URL = https://models.inference.ai.azure.com
  ✅ ROOT → /kaggle/input/datasets/mohammadninadmahmud/medrax-premium/Medrax_premium
  ✅ BENCH_ROOT → /kaggle/input/datasets/mohammadninadmahmud/chestagentbench/chestagentbench
  ✅ MODELS_CHEX → /kaggle/input/datasets/mohammadninadmahmud/models-chexagent/models-chexagent
  ✅ MODELS_CORE → /kaggle/input/datasets/mohammadninadmahmud/models-core/models-core
  ✅ MODELS_LLAVA → /kaggle/input/datasets/mohammadninadmahmud/models-llava-med/models-llava-med
  ✅ MODELS_MEDTOOL → /kaggle/input/datasets/mohammadninadmahmud/models-medical-tools/models-medical-tools
Python 3.12.12 | PyTorch 2.9.0+cu126 | CUDA 12.6
  GPU-0: Tesla T4 — 14.6 GB
  GPU-1: Tesla T4 — 14.6 GB
Logging to: /kaggle/working/medrax_premium_logs/medrax_premium_20260407_130005.json
Target layout — GPU-0: report+cls+seg+LLa

In [3]:

# =====================================================================
# 🔧 Cell 3 — GPU Utilities (free-space guards, checkpoint validation,
#              4-bit overflow budget, max_memory pinning)
# =====================================================================

# ---------------------------------------------------------------------------
# GPU helpers — used by the tool-loading cell below
# ---------------------------------------------------------------------------

def parse_gpu_id(device) -> int:
    """Extract integer GPU index from a device string like 'cuda:1'."""
    s = str(device)
    return int(s.split(":")[-1]) if ":" in s else 0


def check_gpu_free_space(device, required_gb: float, label: str = "model") -> float:
    """Raise RuntimeError if target GPU lacks *required_gb* free VRAM.
    Returns current free GB for logging."""
    if not torch.cuda.is_available():
        print(f"  [{label}] CUDA unavailable — skipping VRAM check")
        return 0.0
    gid = parse_gpu_id(device)
    if gid >= torch.cuda.device_count():
        raise RuntimeError(
            f"[{label}] GPU-{gid} does not exist "
            f"(only {torch.cuda.device_count()} GPU(s) visible)"
        )
    free_b, total_b = torch.cuda.mem_get_info(gid)
    free_gb = free_b / 1024**3
    total_gb = total_b / 1024**3
    print(f"  [{label}] GPU-{gid}: {free_gb:.2f}/{total_gb:.2f} GB free (need ~{required_gb:.1f} GB)")
    if free_gb < required_gb:
        raise RuntimeError(
            f"[{label}] GPU-{gid} only {free_gb:.2f} GB free — "
            f"need ~{required_gb:.1f} GB. Free VRAM or enable 4-bit."
        )
    return free_gb


def gpu_summary() -> str:
    """One-liner per GPU: used / total."""
    lines = []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        used = (total - free) / 1024**3
        total_gb = total / 1024**3
        lines.append(f"  GPU-{i}: {used:.2f}/{total_gb:.2f} GB used")
    return "\n".join(lines)


def validate_checkpoint(path, label: str = "model", required_globs=None):
    """Verify model weight directory exists (+ optional file patterns)."""
    p = Path(path)
    if not p.is_dir():
        raise FileNotFoundError(
            f"[{label}] Checkpoint dir not found: {p}\n"
            f"  Ensure the Kaggle dataset is attached."
        )
    if required_globs:
        for pat in required_globs:
            if not list(p.glob(pat)):
                raise FileNotFoundError(
                    f"[{label}] Required '{pat}' not found in {p}\n"
                    f"  Contents: {sorted(f.name for f in p.iterdir())[:15]}"
                )
    print(f"  [{label}] ✅ {p}")
    return p


def compute_max_memory(device, headroom_gb: float = 1.0, cpu_gb: int = 32):
    """Build max_memory dict that pins quantised model to *device* only.
    Other GPUs get '0GiB' so accelerate never spills there."""
    gid = parse_gpu_id(device)
    mm = {}
    for i in range(torch.cuda.device_count()):
        if i == gid:
            free = torch.cuda.mem_get_info(i)[0] / 1024**3
            budget = max(4, int(free - headroom_gb))
            mm[i] = f"{budget}GiB"
        else:
            mm[i] = "0GiB"
    mm["cpu"] = f"{cpu_gb}GiB"
    return mm


def check_4bit_overflow(device, est_4bit_gb: float, label: str = "model"):
    """Check if 4-bit model fits or needs CPU overflow layers.
    Returns (needs_overflow: bool, free_gb: float)."""
    if not torch.cuda.is_available():
        return True, 0.0
    gid = parse_gpu_id(device)
    free_gb = torch.cuda.mem_get_info(gid)[0] / 1024**3
    scratch = 0.5  # staging / KV-cache scratch
    needs = free_gb < (est_4bit_gb + scratch)
    if needs:
        print(
            f"  [{label}] GPU-{gid}: {free_gb:.2f} GB free but need "
            f"~{est_4bit_gb:.1f}+{scratch} GB → CPU offload for overflow layers"
        )
    else:
        print(
            f"  [{label}] GPU-{gid}: {free_gb:.2f} GB free — "
            f"{est_4bit_gb:.1f} GB 4-bit fits with "
            f"{free_gb - est_4bit_gb:.1f} GB headroom"
        )
    return needs, free_gb


print("✅ GPU utility functions defined")
print(gpu_summary())


✅ GPU utility functions defined
  GPU-0: 0.10/14.56 GB used
  GPU-1: 0.10/14.56 GB used


In [4]:
# =====================================================================
# Cell 4 — Pre-download DeBERTa NLI model for BERT conflict detection
# =====================================================================
# The BERT conflict detector uses microsoft/deberta-base-mnli (~550 MB).
# This cell downloads and caches it so the diagnostic check in Cell 4
# passes and full semantic conflict detection is available.
# Only downloads once — subsequent runs use the HuggingFace cache.

import os, logging as _logging
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers as _tf

_nli_model_name = "microsoft/deberta-base-mnli"

# ── Suppress noisy background safetensors-conversion thread ──────────
# Without an HF token the conversion API call fails (non-fatal but
# pollutes output with a long traceback in a background thread).
# DISABLE_SAFETENSORS_CONVERSION=1 stops transformers from spawning it.
os.environ["DISABLE_SAFETENSORS_CONVERSION"] = "1"
# Suppress "unauthenticated requests" warning from huggingface_hub
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
# Silence the DebertaForSequenceClassification LOAD REPORT (not an error)
_tf.logging.set_verbosity_error()

try:
    # Check if both tokenizer AND model weights are already cached
    AutoTokenizer.from_pretrained(_nli_model_name, local_files_only=True)
    AutoModelForSequenceClassification.from_pretrained(
        _nli_model_name, local_files_only=True
    )
    print(f"✅ {_nli_model_name} already cached — nothing to download")
except Exception:
    print(f"⬇️  Downloading {_nli_model_name} (~550 MB) ...")
    AutoTokenizer.from_pretrained(_nli_model_name)
    AutoModelForSequenceClassification.from_pretrained(_nli_model_name)
    print(f"✅ {_nli_model_name} downloaded and cached successfully")

# Restore normal verbosity for subsequent cells
_tf.logging.set_verbosity_warning()


⬇️  Downloading microsoft/deberta-base-mnli (~550 MB) ...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/557M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ microsoft/deberta-base-mnli downloaded and cached successfully


In [5]:

# =====================================================================
# Cell 5 — Tool Loading (Balanced 2-GPU Layout)
# =====================================================================
#
#   GPU-0 (cuda:0): report (SwinV2x2 fp16)  ~0.5 GB
#                    + classification (DenseNet fp32) ~0.03 GB
#                    + segmentation (PSPNet fp32) ~0.1 GB
#                    + LLaVA-Med 7B (NF4 4-bit) ~4.5 GB
#                    ---------------------------------------- ~5.5 GB
#
#   GPU-1 (cuda:1): CheXagent 8B (NF4 4-bit) ~5 GB
# =====================================================================

from transformers import AutoModelForCausalLM as _AMCLM

# ========== PHASE 0: Validate all model checkpoints ==========
print("=" * 60)
print("Phase 0 -- Validating model checkpoints (Kaggle datasets)")
print("=" * 60)

validate_checkpoint(MODELS_CORE / "swinv2_findings",   label="SwinV2-findings",   required_globs=["*.safetensors", "config.json"])
validate_checkpoint(MODELS_CORE / "swinv2_impression",  label="SwinV2-impression",  required_globs=["*.safetensors", "config.json"])
validate_checkpoint(MODELS_LLAVA,                       label="LLaVA-Med-7B",       required_globs=["config.json"])
validate_checkpoint(MODELS_CHEX,                        label="CheXagent-8B",       required_globs=["config.json"])

print()

# ========== PHASE 1: GPU-0 lightweight tools ====================
print("=" * 60)
print("Phase 1 -- Loading lightweight tools on GPU-0")
print("=" * 60)

check_gpu_free_space(DEVICE_0, required_gb=0.8, label="GPU-0 (lightweight)")
t0 = time.time()

report_tool = ChestXRayReportGeneratorTool(
    cache_dir=str(MODELS_CORE), device=DEVICE_0,
    findings_model_path=str(MODELS_CORE / "swinv2_findings"),
    impression_model_path=str(MODELS_CORE / "swinv2_impression"),
)
xray_classification_tool = ChestXRayClassifierTool(device=DEVICE_0)
segmentation_tool = ChestXRaySegmentationTool(device=DEVICE_0)

free0a, total0 = torch.cuda.mem_get_info(0)
print(f"3 lightweight tools on GPU-0 in {time.time()-t0:.0f}s  "
      f"|  {(total0-free0a)/1024**3:.2f}/{total0/1024**3:.1f} GB")
print()

# ========== PHASE 2: GPU-0 LLaVA-Med 7B (4-bit NF4) ============
print("=" * 60)
print("Phase 2 -- Loading LLaVA-Med 7B on GPU-0 (4-bit NF4)")
print("=" * 60)

_llava_est = 4.5  # estimated 4-bit footprint
check_gpu_free_space(DEVICE_0, required_gb=_llava_est, label="LLaVA-Med-4bit")
needs_overflow_llava, free_before_llava = check_4bit_overflow(
    DEVICE_0, est_4bit_gb=_llava_est, label="LLaVA-Med-4bit",
)

t_llava = time.time()
llava_med_tool = LlavaMedTool(
    model_path=str(MODELS_LLAVA), cache_dir=str(MODELS_LLAVA),
    device=DEVICE_0, load_in_4bit=True,
)

free0b, _ = torch.cuda.mem_get_info(0)
print(f"LLaVA-Med on GPU-0: {(total0-free0b)/1024**3:.2f}/{total0/1024**3:.1f} GB  "
      f"(took {time.time()-t_llava:.0f}s)")
print()

# ========== PHASE 3: GPU-1 CheXagent 8B (4-bit NF4) =============
print("=" * 60)
print("Phase 3 -- Loading CheXagent 8B on GPU-1 (4-bit NF4)")
print("=" * 60)

# Monkey-patch: inject torch_dtype=bf16 so non-quantised layers
# do not materialise in fp32 (~16 GB instead of ~5 GB).
_real_from_pretrained = _AMCLM.from_pretrained

def _patched_from_pretrained(pretrained_model_name_or_path, *args, **kwargs):
    kwargs.setdefault("torch_dtype", torch.bfloat16)
    kwargs.setdefault("low_cpu_mem_usage", True)
    return _real_from_pretrained(pretrained_model_name_or_path, *args, **kwargs)

_AMCLM.from_pretrained = _patched_from_pretrained

_chex_est = 5.0
check_gpu_free_space(DEVICE_1, required_gb=_chex_est, label="CheXagent-4bit")
needs_overflow_chex, free_before_chex = check_4bit_overflow(
    DEVICE_1, est_4bit_gb=_chex_est, label="CheXagent-4bit",
)

t_chex = time.time()
xray_vqa_tool = XRayVQATool(
    model_name=str(MODELS_CHEX), cache_dir=str(MODELS_CHEX),
    device=DEVICE_1, load_in_4bit=True,
)

# Restore original from_pretrained
_AMCLM.from_pretrained = _real_from_pretrained

# ── Hotfix: get_head_mask removed in transformers ≥4.41 ──────────
# CheXagent-8b's QFormer calls self.get_head_mask() which no longer
# exists on PreTrainedModel.  Inject it onto every sub-module that
# has a forward() referencing it.
def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
    if head_mask is not None:
        if head_mask.dim() == 1:
            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
        elif head_mask.dim() == 2:
            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
        if is_attention_chunked:
            head_mask = head_mask.unsqueeze(-1)
    else:
        head_mask = [None] * num_hidden_layers
    return head_mask

# Patch onto QFormer (and any other sub-module that needs it)
_chex_model = xray_vqa_tool.model
for _name, _mod in _chex_model.named_modules():
    if hasattr(type(_mod), 'forward') and not hasattr(_mod, 'get_head_mask'):
        # Only patch modules whose forward actually calls get_head_mask
        import inspect as _insp
        try:
            _src = _insp.getsource(type(_mod).forward)
        except (TypeError, OSError):
            _src = ""
        if 'get_head_mask' in _src:
            import types as _types
            _mod.get_head_mask = _types.MethodType(_get_head_mask, _mod)
            print(f"  [Hotfix] Patched get_head_mask onto {_name}")
print("  [Hotfix] get_head_mask patch applied ✅")

free1, total1 = torch.cuda.mem_get_info(1)
print(f"CheXagent on GPU-1: {(total1-free1)/1024**3:.2f}/{total1/1024**3:.1f} GB  "
      f"(took {time.time()-t_chex:.0f}s)")
print()

# ========== PHASE 4b: Monkey-patch BLIP-2 prompt ==================
# CheXagent-8b (BLIP-2 arch) gives very terse output (1-2 findings)
# with bare prompts.  We wrap the user prompt with an enumeration
# instruction so it lists ALL visible pathologies — critical for the
# conflict-resolution pipeline to have enough findings to compare.
# -------------------------------------------------------------------
if xray_vqa_tool._is_blip2:
    _orig_gen_inner = xray_vqa_tool._generate_response_inner

    def _patched_generate_response_inner(
        image_paths, prompt, max_new_tokens,
        do_sample=False, temperature=1.0, top_p=1.0,
    ):
        _enum_prefix = (
            "You are an expert radiologist. Carefully examine this chest X-ray "
            "and provide a detailed, structured analysis. "
            "List ALL visible findings, abnormalities, and relevant negative findings. "
            "For each finding state: (1) the pathology name, (2) its location, "
            "(3) severity (mild/moderate/severe), and (4) your confidence. "
            "Do not summarize — enumerate every observation.\n\n"
        )
        return _orig_gen_inner(
            image_paths, _enum_prefix + prompt, max_new_tokens,
            do_sample=do_sample, temperature=temperature, top_p=top_p,
        )

    xray_vqa_tool._generate_response_inner = _patched_generate_response_inner
    print("  [Patch] BLIP-2 prompt enhanced for detailed pathology enumeration ✅")
else:
    print("  [Info] Qwen-VL arch — no BLIP-2 prompt patch needed")

# ========== PHASE 4c: Fix TOOL_EXPERTISE tool names ==================
# ConflictResolver.TOOL_EXPERTISE uses wrong names for localization.
# Actual tool names: "chest_xray_segmentation", "xray_phrase_grounding"
from medrax_premium.agent.conflict_resolution import ConflictResolver
ConflictResolver.TOOL_EXPERTISE["localization"] = {
    "primary": "chest_xray_segmentation",
    "fallback": "xray_phrase_grounding",
    "description": "Precise anatomical location",
}
print("  [Patch] ConflictResolver.TOOL_EXPERTISE localization names fixed ✅")

# ========== PHASE 5: Cleanup + Summary =============================
gc.collect(); torch.cuda.empty_cache()

ALL_TOOLS = [
    report_tool, xray_classification_tool, segmentation_tool,
    llava_med_tool, xray_vqa_tool,
]

free0_final, _ = torch.cuda.mem_get_info(0)
free1_final, _ = torch.cuda.mem_get_info(1)
print("=" * 60)
print(f"{len(ALL_TOOLS)} tools loaded -- balanced 2-GPU layout")
print(f"   GPU-0 (report+cls+seg+LLaVA): "
      f"{(total0-free0_final)/1024**3:.2f}/{total0/1024**3:.1f} GB")
print(f"   GPU-1 (CheXagent):             "
      f"{(total1-free1_final)/1024**3:.2f}/{total1/1024**3:.1f} GB")
if needs_overflow_llava:
    print("   LLaVA-Med used CPU offload for some layers")
if needs_overflow_chex:
    print("   CheXagent used CPU offload for some layers")
print("=" * 60)

# ========== DIAGNOSTIC: BERT NLI availability check =================
# DeBERTa needs to be either cached locally or internet-downloadable.
# On Kaggle without internet, it will silently fall back to rule-based.
_bert_available = False
try:
    from transformers import AutoTokenizer
    _test_tok = AutoTokenizer.from_pretrained(
        "microsoft/deberta-base-mnli",
        local_files_only=True,  # Don't download — just check cache
    )
    _bert_available = True
    del _test_tok
except Exception:
    pass

if _bert_available:
    print("✅ BERT NLI model (deberta-base-mnli) found in cache — full semantic conflict detection")
else:
    print("⚠️  BERT NLI model NOT cached — will use rule-based conflict detection only")
    print("   (Rule-based still catches presence/absence conflicts; GACL catches anatomical inconsistencies)")
    print("   To enable BERT: pre-download deberta-base-mnli or add internet access")


# =====================================================================
# Agent + Runner
# =====================================================================

# ── FIX: Pre-warm the BERT detector once so every Agent reuses it ──
# Without this, each get_agent() call lazily reloads the 550 MB model.
from medrax_premium.agent.conflict_resolution import ConflictDetector
_warmup_detector = ConflictDetector(sensitivity=0.25, use_bert=_bert_available)
if _bert_available:
    _warmup_detector.bert_device = "cpu"
    _ = _warmup_detector.bert_detector  # triggers load once
    print("✅ BERT conflict detector pre-warmed (shared across agents)")


def get_agent():
    """Create a fresh Premium agent with conflict resolution enabled."""
    prompts = load_prompts_from_file(str(PROMPT_FILE))
    prompt = prompts["MEDICAL_ASSISTANT"]

    # Resolve API key — Cell 3 already guaranteed this is set;
    # we retrieve it again here in case get_agent() is called standalone.
    _key = os.environ.get("OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not _key:
        raise EnvironmentError(
            "OPENAI_API_KEY missing. Run Cell 3 first, or set the Kaggle Secret."
        )

    model = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=temperature,
        top_p=0.95,
        api_key=_key,
        base_url=os.environ.get("OPENAI_BASE_URL") or None,
    )
    agent = Agent(
        model, tools=ALL_TOOLS, log_tools=True,
        log_dir=str(LOG_DIR / "tool_logs"),
        system_prompt=prompt, checkpointer=MemorySaver(),
        enable_conflict_resolution=True,
        conflict_sensitivity=0.25, deferral_threshold=0.6,
    )
    # Pin BERT conflict detector to CPU to avoid GPU-0 OOM (DeBERTa ~400MB)
    if hasattr(agent, "conflict_detector") and agent.conflict_detector.use_bert:
        agent.conflict_detector.bert_device = "cpu"
    thread = {"configurable": {"thread_id": str(uuid.uuid4())}}
    return agent, thread


def run_medrax(agent, thread, prompt, _max_retries=5):
    """Run a single turn through the ReAct agent with rate-limit retry.

    NOTE: We do NOT send images to the LLM.  The original MedRAX design
    (see interface.py L113) passes only the file path as text; the local
    GPU tools (LLaVA-Med, CheXagent, classifier, segmentation) open the
    file themselves.  Embedding base64 images in the HumanMessage blows
    up the HTTP payload to megabytes → 429 on GitHub Models.
    """
    messages = [
        HumanMessage(content=prompt)
    ]

    for attempt in range(_max_retries):
        try:
            final_response = None
            for event in agent.workflow.stream({"messages": messages}, thread):
                for v in event.values():
                    final_response = v

            if isinstance(final_response, dict) and final_response.get("messages"):
                final_response = final_response["messages"][-1].content.strip()
            elif final_response is not None:
                final_response = str(final_response).strip()

            agent_state = agent.workflow.get_state(thread)
            return final_response or "", str(agent_state)

        except Exception as e:
            err_str = str(e)

            # Retry on rate-limit (429) or token-limit (413) with exponential backoff
            if any(sig in err_str.lower() for sig in (
                "429", "413", "ratelimitreached",
                "tokens_limit_reached", "rate limit", "rate_limit",
                "token", "too large", "payload too large",
            )):
                wait = min(2 ** attempt * 10, 120)  # 10s, 20s, 40s, 80s, 120s
                print(f"  ⏳ Rate/token limited (attempt {attempt+1}/{_max_retries}), waiting {wait}s...")
                print(f"     Error: {err_str[:300]}")  # PRINT THE ACTUAL ERROR
                time.sleep(wait)
                continue

            raise  # Non-rate-limit errors bubble up immediately

    raise RuntimeError(f"Rate limit exceeded after {_max_retries} retries")


Phase 0 -- Validating model checkpoints (Kaggle datasets)
  [SwinV2-findings] ✅ /kaggle/input/datasets/mohammadninadmahmud/models-core/models-core/swinv2_findings
  [SwinV2-impression] ✅ /kaggle/input/datasets/mohammadninadmahmud/models-core/models-core/swinv2_impression
  [LLaVA-Med-7B] ✅ /kaggle/input/datasets/mohammadninadmahmud/models-llava-med/models-llava-med
  [CheXagent-8B] ✅ /kaggle/input/datasets/mohammadninadmahmud/models-chexagent/models-chexagent

Phase 1 -- Loading lightweight tools on GPU-0
  [GPU-0 (lightweight)] GPU-0: 14.46/14.56 GB free (need ~0.8 GB)


Loading weights:   0%|          | 0/300 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/300 [00:00<?, ?it/s]

If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt -O /root/.torchxrayvision/models_data/nih-pc-chex-mimic_ch-google-openi-kaggle-densenet121-d121-tw-lr001-rot45-tr15-sc15-seed0-best.pt`
[██████████████████████████████████████████████████]
If this fails you can run `wget https://github.com/mlmed/torchxrayvision/releases/download/v1/pspnet_chestxray_best_model_4.pth -O /root/.torchxrayvision/models_data/pspnet_chestxray_best_model_4.pth`
[██████████████████████████████████████████████████]
3 lightweight tools on GPU-0 in 8s  |  0.63/14.6 GB

Phase 2 -- Loading LLaVA-Med 7B on GPU-0 (4-bit NF4)
  [LLaVA-Med-4bit] GPU-0: 13.93/14.56 GB free (need ~4.5 GB)
  [LLaVA-Med-4bit] GPU-0: 13.93 GB free — 4.5 GB 4-bit fits with 9.4 GB headroom
  [LLaVA] max_memory = {0: '12GiB', 1: '0GiB', 'cpu': '24GiB'}


config.json: 0.00B [00:00, ?B/s]

...

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


  [CheXagent] BLIP-2 architecture detected — using AutoProcessor
  [Hotfix] Patched get_head_mask onto qformer
  [Hotfix] get_head_mask patch applied ✅
CheXagent on GPU-1: 4.66/14.6 GB  (took 63s)

  [Patch] BLIP-2 prompt enhanced for detailed pathology enumeration ✅
  [Patch] ConflictResolver.TOOL_EXPERTISE localization names fixed ✅
5 tools loaded -- balanced 2-GPU layout
   GPU-0 (report+cls+seg+LLaVA): 5.13/14.6 GB
   GPU-1 (CheXagent):             4.60/14.6 GB
✅ BERT NLI model (deberta-base-mnli) found in cache — full semantic conflict detection
Loading NLI model: microsoft/deberta-base-mnli


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

DebertaForSequenceClassification LOAD REPORT from: microsoft/deberta-base-mnli
Key    | Status     |  | 
-------+------------+--+-
config | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ BERT conflict detector loaded successfully
✅ BERT conflict detector pre-warmed (shared across agents)


In [6]:
# ===============================
# 🚀 Supervisor Demo Launcher
# ===============================
# Run this after Cell 4 so the models are loaded before the demo starts.

%cd {ROOT}
!python main.py

/kaggle/input/datasets/mohammadninadmahmud/medrax-premium/Medrax_premium
Starting server...
Traceback (most recent call last):
  File "/kaggle/input/datasets/mohammadninadmahmud/medrax-premium/Medrax_premium/main.py", line 182, in <module>
    agent, tools_dict = initialize_agent(
                        ^^^^^^^^^^^^^^^^^
  File "/kaggle/input/datasets/mohammadninadmahmud/medrax-premium/Medrax_premium/main.py", line 117, in initialize_agent
    XRayPhraseGroundingTool, gpu1_mgr,
    ^^^^^^^^^^^^^^^^^^^^^^^
NameError: name 'XRayPhraseGroundingTool' is not defined
[W407 13:04:12.292211794 AllocatorConfig.cpp:28] Warning: PYTORCH_CUDA_ALLOC_CONF is deprecated, use PYTORCH_ALLOC_CONF instead (function operator())
